In [ ]:
class Truck(Env):
    """
    A custom environment representing a car that can move in four directions,
    load cargo at a specific location, and unload it at another location, within a 2D grid.

    Attributes:
        EAST, WEST, SOUTH, NORTH (int): Possible actions for movement in the grid.
        LOAD, UNLOAD (int): Actions for loading and unloading cargo.
        actions_as_text (list): A list of action names for reference.
        NOTALLOWED, ALLOWED, BARRIER (int): Represent different types of grid cells.
        anz_rows, anz_cols (int): Dimensions of the playing field grid.
        observation_space (MultiDiscrete): The observation space, consisting of the grid dimensions and load status.
        action_space (Discrete): The action space, consisting of movement and load/unload actions.
        playing_field (np.ndarray): A matrix representing the playing field (0=not allowed, 1=allowed, 2=barrier).
        row, col (int): The current position of the car in the grid.
        act_state (np.ndarray): The current state of the car (position and load status).
        drive_length (int): The maximum number of steps the car can take.
        num_steps (int): The number of steps taken so far.
        car_has_load (bool): Whether the car is currently carrying a load.
        model_file, dqn_file (str): File paths for saving/loading models.
    """

    def __init__(self):
        """
        Initializes the Car environment. Sets up the grid, action/observation spaces,
        and places the car in a random starting position.
        """
        # Define possible actions for the car
        self.EAST = 0
        self.WEST = 1
        self.SOUTH = 2
        self.NORTH = 3
        self.LOAD = 4
        self.UNLOAD = 5

        # List of action names
        self.actions_as_text = ["East", "West",
                                "South", "North", "Load", "Unload"]

        # The playing field: constants for grid states
        self.NOTALLOWED = 0  # Areas the car cannot enter
        self.ALLOWED = 1  # Valid paths the car can follow
        self.BARRIER = 2  # Barriers that block the car's movement
        self.anz_rows = 15  # Number of rows in the grid
        self.anz_cols = 15  # Number of columns in the grid

        # Define the observation space: rows, columns, and load status
        self.observation_space = MultiDiscrete(
            [self.anz_rows, self.anz_cols, 2])

        # Define the action space: 6 actions (4 directions + load/unload)
        self.action_space = Discrete(6)

        # Create a matrix for the playing field (all cells start as NOTALLOWED)
        self.playing_field = np.zeros((self.anz_rows, self.anz_cols))
        self.define_allowed_path()  # Define paths and barriers in the grid

        # Set the truck's random starting position
        self.row = random.randint(0, self.anz_rows - 1)
        self.col = random.randint(0, self.anz_cols - 1)

        # Initialize the current state (position and load status)
        self.act_state = np.array([self.row, self.col, 1])

        # Define parameters for tracking steps and loading
        self.drive_length = 150  # Maximum allowed steps
        self.num_steps = 0  # Steps taken so far
        self.car_has_load = False  # Whether the car has a load

        # File paths for saving/loading models
        self.model_file = f"./Car_model/car_model_2D_{self.anz_cols}x{self.anz_rows}"
        self.dqn_file = f"./Car_model/dqn_2D_{self.anz_cols}x{self.anz_rows}.h5"

    def define_allowed_path(self):
        """
        Defines the valid paths and barriers in the grid, creating an inner rectangle
        with detours around the barriers. Also defines the load and unload positions.
        """
        # Define the first inner rectangle
        inner_row_start = 1
        inner_row_end = self.anz_rows - 2
        inner_col_start = 1
        inner_col_end = self.anz_cols - 2

        # Mark the allowed path for the inner rectangle
        self.playing_field[inner_row_start,
                           inner_col_start:inner_col_end + 1] = self.ALLOWED  # Top
        self.playing_field[inner_row_end,
                           inner_col_start:inner_col_end + 1] = self.ALLOWED  # Bottom
        self.playing_field[inner_row_start:inner_row_end +
                           1, inner_col_start] = self.ALLOWED  # Left
        self.playing_field[inner_row_start:inner_row_end +
                           1, inner_col_end] = self.ALLOWED  # Right

        # Define barriers in the middle of each side of the rectangle
        middle_col = self.anz_cols // 2
        middle_row = self.anz_rows // 2

        # Block the barriers
        self.playing_field[inner_row_start, middle_col -
                           1:middle_col + 1] = self.BARRIER  # Top
        self.playing_field[inner_row_end, middle_col -
                           1:middle_col + 1] = self.BARRIER  # Bottom
        self.playing_field[middle_row - 1:middle_row +
                           1, inner_col_start] = self.BARRIER  # Left
        self.playing_field[middle_row - 1:middle_row +
                           1, inner_col_end] = self.BARRIER  # Right

        # Create detours around the barriers
        self.playing_field[inner_row_start + 1, middle_col -
                           2:middle_col + 2] = self.ALLOWED  # Top detour
        self.playing_field[inner_row_end - 1, middle_col -
                           2:middle_col + 2] = self.ALLOWED  # Bottom detour
        self.playing_field[middle_row - 2:middle_row + 2,
                           inner_col_start + 1] = self.ALLOWED  # Left detour
        self.playing_field[middle_row - 2:middle_row + 2,
                           inner_col_end - 1] = self.ALLOWED  # Right detour

        # Set load and unload positions
        self.LOAD_POS = [inner_row_start, inner_col_end]
        self.UNLOAD_POS = [inner_row_end, inner_col_start]

    ######################################################
    # Perform one step in the environment
    ######################################################
    def step(self, action):
        """
        Executes the given action in the environment, updates the truck's position or load status,
        and returns the new state, reward, whether the episode is done, and additional info.

        Args:
            action (int): The action to be performed (move or load/unload).

        Returns:
            tuple: (new_state (np.ndarray), reward (int), done (bool), info (dict))
        """
        self.drive_length -= 1  # Decrease the number of steps remaining
        self.num_steps += 1  # Increment the step counter
        done = False
        action_reward = -8  # Default negative reward for non-optimal actions

        # Define the boundaries based on the current grid size
        field_row_end = self.anz_rows - 1
        field_col_end = self.anz_cols - 1

        # Movement logic and rewards based on the action
        if action == self.EAST and self.col < field_col_end-1:
            self.col += 1
            if self.playing_field[self.row, self.col] == self.ALLOWED:
                action_reward = -1

        elif action == self.WEST and self.col > 1:
            self.col -= 1
            if self.playing_field[self.row, self.col] == self.ALLOWED:
                action_reward = -1

        elif action == self.SOUTH and self.row < field_row_end-1:
            self.row += 1
            if self.playing_field[self.row, self.col] == self.ALLOWED:
                action_reward = -1

        elif action == self.NORTH and self.row > 1:
            self.row -= 1
            if self.playing_field[self.row, self.col] == self.ALLOWED:
                action_reward = -1

        # Load and unload logic
        elif action == self.LOAD:
            if [self.row, self.col] == self.LOAD_POS and not self.truck_has_load:
                self.truck_has_load = True
                action_reward = 70
            else:
                action_reward = -5

        elif action == self.UNLOAD:
            if [self.row, self.col] == self.UNLOAD_POS and self.truck_has_load:
                self.truck_has_load = False
                action_reward = 70
            else:
                action_reward = -5

        # Invalid movement (out of bounds or into barriers)
        else:
            if action == self.EAST:
                self.col = min(self.col + 1, field_col_end)
            elif action == self.WEST:
                self.col = max(self.col - 1, 0)
            elif action == self.NORTH:
                self.row = max(self.row - 1, 0)
            elif action == self.SOUTH:
                self.row = min(self.row + 1, field_row_end)

        # Check if the episode is finished
        if self.drive_length < 0:
            done = True

        # Update the current state and return it along with the reward and info
        self.act_state[0] = self.row
        self.act_state[1] = self.col
        self.act_state[2] = self.truck_has_load
        info = {}
        return self.act_state, action_reward, done, info

    ############################
    # Reset/Set the environment
    ############################
    def reset(self):
        """
        Resets the environment to a random starting position. The truck's position is set to a 
        random point within the grid, and the drive length and load status are reset to their 
        initial values. 

        Returns:
            np.ndarray: The new state of the environment, consisting of the truck's position 
                        (row, column) and whether it has a load.
        """
        # Reset to random position within the allowed path
        self.row = random.randint(0, self.anz_rows - 1)
        self.col = random.randint(0, self.anz_cols - 1)

        # Reset the state of the truck (position and load status)
        self.act_state[0] = self.row
        self.act_state[1] = self.col
        self.drive_length = 150  # Reset the number of remaining steps
        self.truck_has_load = False  # The truck starts without a load
        # Update the load status in the state
        self.act_state[2] = self.truck_has_load

        return self.act_state  # Return the reset state

    ######################################################
    # Render the environment and truck movement/load visually
    ######################################################
    def render(self, episode, reward, episode_score):
        """
        Renders the environment using Pygame. This method draws the current state of the grid,
        including barriers, allowed paths, load/unload positions, and the truck itself. It also
        displays information such as the number of steps remaining, the reward for the current 
        step, and the cumulative episode score.

        Args:
            episode (int): The current episode number.
            reward (int): The reward received for the current step.
            episode_score (int): The cumulative score for the current episode.

        Rendering Details:
            - The grid is displayed with each cell representing an area of the environment.
            - The truck is shown in green (with load) or dark green (without load).
            - Load/unload positions are marked with distinct colors.
            - Text at the top of the window shows the remaining steps, the step reward, and the episode score.
        """
        # Define constants for colors (RGB)
        GRAY = (128, 128, 128)  # Non-valid area (0)
        WHITE = (255, 255, 255)  # Valid path (1)
        RED = (255, 0, 0)  # Barriers (2)
        GREEN = (50, 255, 50)  # truck without load
        DARK_GREEN = (0, 140, 0)  # truck with load
        BLACK = (0, 0, 0)  # Background color
        LOAD = (255, 255, 0)  # Load position
        UNLOAD = (255, 0, 255)  # Unload position

        # Define grid cell size and screen size
        CELL_SIZE = 40  # Size of each grid cell (in pixels)
        ROWS, COLS = self.anz_rows, self.anz_cols  # Grid dimensions
        WINDOW_WIDTH = COLS * CELL_SIZE  # Total window width
        TEXT_HEIGHT = 35  # Height of the text's
        WINDOW_HEIGHT = ROWS * CELL_SIZE + 3 * TEXT_HEIGHT  # Total window height

        # Initialize Pygame if it hasn't been done yet
        if not hasattr(self, 'screen'):
            pygame.init()  # Initialize the Pygame engine
            self.screen = pygame.display.set_mode(
                (WINDOW_WIDTH, WINDOW_HEIGHT))  # Set up the window
            self.clock = pygame.time.Clock()  # Set up the clock for frame rate control
            pygame.font.init()  # Initialize font module
            # Use the default font (size 36)
            self.font = pygame.font.SysFont(None, 36)

        pygame.display.set_caption(
            f"Run the truck, Episode {episode}")  # Set window caption

        # Handle Pygame events (even though we don't need them right now)
        for event in pygame.event.get():
            if event.type == pygame.QUIT:  # Handle the close window event
                pygame.quit()

        # Clear the screen (fill it with black background)
        self.screen.fill(BLACK)

        # Display text for steps remaining, step reward, and episode score
        steps_text = self.font.render(
            f"Steps remaining: {self.drive_length}", True, WHITE)
        reward_text = self.font.render(
            f"Step reward: {reward}", True, WHITE)
        episode_reward_text = self.font.render(
            f"Episode reward: {episode_score}", True, WHITE)

        # Draw the text at the top of the window
        self.screen.blit(steps_text, (10, 10))
        self.screen.blit(reward_text, (10, 40))
        self.screen.blit(episode_reward_text, (10, 70))

        # Render the grid with the playing field, truck, and load/unload locations
        for row in range(ROWS):
            for col in range(COLS):
                # Get the value of the current grid cell
                value = int(self.playing_field[row, col])

                # Choose the color based on the value of the cell
                if value == self.NOTALLOWED:
                    color = GRAY  # Non-valid area
                elif value == self.ALLOWED:
                    color = WHITE  # Valid path
                elif value == self.BARRIER:
                    color = RED  # Barrier

                # Load and unload cells are colored distinctly
                if [row, col] == self.LOAD_POS:
                    color = LOAD  # Load position
                elif [row, col] == self.UNLOAD_POS:
                    color = UNLOAD  # Unload position

                # If this is the truck's current position, color it accordingly
                if row == self.row and col == self.col:
                    if self.truck_has_load:
                        color = DARK_GREEN  # truck with load
                    else:
                        color = GREEN  # truck without load

                # Draw each cell as a rectangle (offset by TEXT_HEIGHT)
                pygame.draw.rect(self.screen, color, pygame.Rect(
                    col * CELL_SIZE, row * CELL_SIZE + 3 * TEXT_HEIGHT, CELL_SIZE, CELL_SIZE))

                # Optionally, draw grid lines for better visibility
                pygame.draw.rect(self.screen, BLACK, pygame.Rect(
                    col * CELL_SIZE, row * CELL_SIZE + 3 * TEXT_HEIGHT, CELL_SIZE, CELL_SIZE), 1)

        # Update the display to show the rendered grid
        pygame.display.flip()

        # Optionally, control the frame rate for smoother rendering
        self.clock.tick(6)